# Chapter 15: the four Kestrel trust checks

Each check ties a chart to a number the brewer already knows and prints PASS or FAIL. One FAIL is planted:
tank F11 drops out for six hours on 14 April, so its sensor count comes up short. Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

In [ ]:
results = []
def check(name, passed, detail):
    results.append([name, "PASS" if passed else "FAIL", detail])

vol = load("Volume").set_index("Month").loc["2026-03-01"]
loss = vol.CastHL - vol.DispatchedHL
check("Loss waterfall", loss == 370 and round(100 * loss / vol.CastHL, 2) == 4.25,
      f"{vol.CastHL:,} - {vol.DispatchedHL:,} = {loss} hL, {100 * loss / vol.CastHL:.2f} percent")

fill = load("Fill")
used = fill[fill.FillId.str.contains("-26-")].ResidenceDays.sum()
check("Tank occupancy", abs(used - 4650) <= 50 and round(100 * used / 5475) == 85,
      f"{used:,} of 5,475 tank days, {100 * used / 5475:.0f} percent")

In [ ]:
lab = load("Lab", parse_dates=["SampledAt"])
hg = load("HourlyGravity")
f = load("Fill", parse_dates=["HourZero"])[["FillId", "HourZero"]]
ls = lab[(lab.Analyte == "Apparent extract") & lab.FillId.isin(hg.FillId)].merge(f, on="FillId")
ls["BatchHour"] = ((ls.SampledAt - ls.HourZero) / pd.Timedelta(hours=1)).astype(int)
gap = (ls.merge(hg, on=["FillId", "BatchHour"]).eval("Gravity - Value")).mean()
check("Cellar density offset", round(gap, 1) == 0.1, f"inline minus lab = {gap:.2f} degrees Plato over {len(ls)} samples, shown not hidden")

rd = load("Readings")
per_tank = rd[rd.Tag == "Temp"].groupby("Tank").size()
short = per_tank[per_tank < 1440]
check("Sensor count", short.empty, "1,440 a day on every tank" if short.empty else
      "; ".join(f"{t} has {n:,} rows, {(1440 - n) / 60:.0f} h missing" for t, n in short.items()))

print(pd.DataFrame(results, columns=["check", "result", "detail"]).to_string(index=False))

**What to look at.** A pass ties the chart to a number the brewer knows. A fail is a dropout to chase before anyone reads
the tank wall. Compare against an independent number too, because a check can pass on the same bad source.